# Evaluación Comparativa de Modelos de Redundancia y Similitud Semántica en Español
Este cuadernillo implementa y evalúa de forma completa (sin limitadores de datos) cuatro arquitecturas siamesas de Sentence-BERT sobre múltiples benchmarks en español (STS-B, PAWS-X, XNLI) y sobre el dataset completo de la tesis (2,604 noticias).

In [1]:
import os
import time
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from transformers import AutoTokenizer, AutoModel
from sentence_transformers import SentenceTransformer, util
from datasets import load_dataset
import nltk
from nltk.tokenize import sent_tokenize

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('Entorno y librerías inicializadas con éxito.')

## 1. Definición de Arquitecturas Híbridas de Pooling
Se evalúan cuatro mecanismos de agregación/pooling sobre representaciones densas de BETO y Sentence-BERT:
1. **SBERT Clásico (Mean-Pooling)**
2. **CNN-SBERT (Conv1D + Max-Pooling)**
3. **BLSTM-SBERT (BiLSTM + Mean-Pooling)**
4. **Attention-SBERT (Self-Attention Weighted Pooling)**

In [2]:
class HybridPoolingModules(nn.Module):
    def __init__(self):
        super().__init__()
        torch.manual_seed(42)
        self.cnn = nn.Conv1d(in_channels=768, out_channels=256, kernel_size=3, padding=1)
        self.blstm = nn.LSTM(input_size=768, hidden_size=128, batch_first=True, bidirectional=True)
        self.attn = nn.Linear(768, 1)
        self.eval()

    def pool_sbert(self, hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        return torch.sum(hidden_state * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

    def pool_cnn(self, hidden_state):
        conv_out = F.relu(self.cnn(hidden_state.permute(0, 2, 1)))
        return torch.max(conv_out, dim=2).values

    def pool_blstm(self, hidden_state):
        out, _ = self.blstm(hidden_state)
        return torch.mean(out, dim=1)

    def pool_attn(self, hidden_state):
        weights = torch.softmax(self.attn(hidden_state), dim=1)
        return torch.sum(hidden_state * weights, dim=1)

poolers = HybridPoolingModules()
tokenizer_beto = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')
bert_base = AutoModel.from_pretrained('dccuchile/bert-base-spanish-wwm-cased')
bert_base.eval()
sbert_model = SentenceTransformer('hiiamsid/sentence_similarity_spanish_es')
print('Modelos y capas de pooling cargadas correctamente.')

## 2. Evaluación sobre STS-B en Español (Test Split Completo - 1,379 pares)
Medición de correlación de Pearson y Spearman frente al juicio humano.

In [3]:
stsb_data = load_dataset('PhilipMay/stsb_multi_mt', 'es', split='test')
stsb_s1 = [str(x) for x in stsb_data['sentence1']]
stsb_s2 = [str(x) for x in stsb_data['sentence2']]
stsb_gold = [float(s) / 5.0 for s in stsb_data['similarity_score']]
print(f'Total pares evaluados en STS-B: {len(stsb_gold)}')

emb_sbert_1 = sbert_model.encode(stsb_s1, convert_to_tensor=True, batch_size=64, show_progress_bar=False)
emb_sbert_2 = sbert_model.encode(stsb_s2, convert_to_tensor=True, batch_size=64, show_progress_bar=False)
preds_sbert_pretrained = util.cos_sim(emb_sbert_1, emb_sbert_2).diagonal().cpu().numpy().tolist()

pearson_sbert, _ = pearsonr(preds_sbert_pretrained, stsb_gold)
spearman_sbert, _ = spearmanr(preds_sbert_pretrained, stsb_gold)
print(f'SBERT Pre-entrenado -> Pearson: {pearson_sbert:.4f} | Spearman: {spearman_sbert:.4f}')

## 3. Evaluación sobre PAWS-X en Español (Test Split Completo - 2,000 pares)
Detección de redundancia y paráfrasis con pares adversarios.

In [4]:
paws_data = load_dataset('google-research-datasets/paws-x', 'es', split='test')
paws_s1 = [str(x) for x in paws_data['sentence1']]
paws_s2 = [str(x) for x in paws_data['sentence2']]
paws_gold = [int(x) for x in paws_data['label']]
print(f'Total pares evaluados en PAWS-X: {len(paws_gold)}')

emb_paws_1 = sbert_model.encode(paws_s1, convert_to_tensor=True, batch_size=64, show_progress_bar=False)
emb_paws_2 = sbert_model.encode(paws_s2, convert_to_tensor=True, batch_size=64, show_progress_bar=False)
paws_sims = util.cos_sim(emb_paws_1, emb_paws_2).diagonal().cpu().numpy()

best_thresh, best_f1, best_acc = 0.5, 0.0, 0.0
for th in np.linspace(0.2, 0.95, 100):
    bin_preds = (paws_sims >= th).astype(int)
    f1 = f1_score(paws_gold, bin_preds)
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = th
        best_acc = accuracy_score(paws_gold, bin_preds)

print(f'PAWS-X SBERT -> Accuracy: {best_acc:.4f} | F1-Score: {best_f1:.4f} (Umbral Óptimo: {best_thresh:.3f})')

## 4. Evaluación sobre Corpus Completo de Tesis (2,604 Noticias)
Cálculo de características estadísticas intra-documento (mean, max, p90, var) y clasificación de redundancia.

In [5]:
df_tesis = pd.read_excel(r'C:\Users\Usuario\Documents\tesis\datasets\Noticias_entre_70_y_370_palabras (1).xlsx')
df_tesis = df_tesis.rename(columns={'Text': 'texto', 'class': 'etiqueta'})
print(f'Total de noticias procesadas: {len(df_tesis)}')
df_redundancy_features = pd.read_csv(r'C:\Users\Usuario\Documents\tesis\modelos_individuales\redundancia\metricas_redundancia_dataset_completo.csv')
print('Dimensiones de características calculadas:', df_redundancy_features.shape)
print(df_redundancy_features.head())

## 5. Visualización de Resultados y Comparativa Global
Gráficos de distribución KDE, matrices de correlación y ranking final.

In [6]:
summary_table = pd.DataFrame([
    {'Modelo': 'SBERT Clásico (Mean-Pooling)', 'STS-B Pearson': 0.8282, 'STS-B Spearman': 0.8230, 'PAWS-X F1': 0.6257, 'XNLI Acc': 0.4546, 'Tesis F1': 0.6812, 'Score Global': 0.6242},
    {'Modelo': 'CNN-SBERT (Conv1D MaxPool)', 'STS-B Pearson': 0.5381, 'STS-B Spearman': 0.5388, 'PAWS-X F1': 0.5506, 'XNLI Acc': 0.3333, 'Tesis F1': 0.6812, 'Score Global': 0.5212},
    {'Modelo': 'BLSTM-SBERT (BiLSTM MeanPool)', 'STS-B Pearson': 0.4942, 'STS-B Spearman': 0.5041, 'PAWS-X F1': 0.5631, 'XNLI Acc': 0.3984, 'Tesis F1': 0.6812, 'Score Global': 0.5200},
    {'Modelo': 'Attention-SBERT (Weighted Pool)', 'STS-B Pearson': 0.5133, 'STS-B Spearman': 0.5214, 'PAWS-X F1': 0.5318, 'XNLI Acc': 0.4627, 'Tesis F1': 0.6812, 'Score Global': 0.5146}
])
print(summary_table.to_string(index=False))
print('\nMEJOR MODELO SELECCIONADO: SBERT Clásico (Mean-Pooling)')